# Assignment 6: Data Generation using Modelling and Simulation for Machine Learning

## Objective
To generate a dataset using a simulation tool (SimPy) and apply various Machine Learning models to predict the outcome.

## Simulation Scenario: Single-Server Queue (M/M/1)
We obtain data by simulating a bank teller system where customers arrive randomly and are served one by one.

**Parameters:**
- **Arrival Rate:** Rate at which customers arrive.
- **Service Rate:** Rate at which the teller serves customers.
- **Target:** Average Wait Time for customers in the queue.

In [ ]:
!pip install simpy pandas scikit-learn matplotlib seaborn

In [ ]:
import simpy
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
def bank_teller_simulation(env, mean_arrival, mean_service, wait_times):
    """
    Simulates a Bank Teller system.
    env: SimPy environment
    mean_arrival: Average time between arrivals
    mean_service: Average time to serve a customer
    wait_times: List to store waiting times of customers
    """
    teller = simpy.Resource(env, capacity=1)

    def customer(env, name, teller, mean_service, wait_times):
        arrival_time = env.now
        request = teller.request()
        yield request
        wait = env.now - arrival_time
        wait_times.append(wait)
        yield env.timeout(random.expovariate(1.0 / mean_service))
        teller.release(request)

    i = 0
    while True:
        yield env.timeout(random.expovariate(1.0 / mean_arrival))
        i += 1
        env.process(customer(env, f'Customer {i}', teller, mean_service, wait_times))

In [ ]:
def run_single_simulation(mean_arrival, mean_service, simulation_time=100):
    """
    Runs a single simulation instance and returns the average wait time.
    """
    env = simpy.Environment()
    wait_times = []
    env.process(bank_teller_simulation(env, mean_arrival, mean_service, wait_times))
    env.run(until=simulation_time)
    
    if len(wait_times) > 0:
        return sum(wait_times) / len(wait_times)
    return 0.0

In [ ]:
data = []
num_simulations = 1000

print(f"Generating {num_simulations} simulations...")

for _ in range(num_simulations):
    arrival_val = random.uniform(1.0, 5.0)
    service_val = random.uniform(0.5, 4.0)

    avg_wait = run_single_simulation(arrival_val, service_val)
    
    data.append({
        'Mean_Interarrival_Time': arrival_val,
        'Mean_Service_Time': service_val,
        'Average_Wait_Time': avg_wait
    })

df = pd.DataFrame(data)
print("Data generation complete.")
df.head()

In [ ]:
X = df[['Mean_Interarrival_Time', 'Mean_Service_Time']]
y = df['Average_Wait_Time']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "AdaBoost": AdaBoostRegressor(),
    "Support Vector Regressor": SVR(),
    "MLP Regressor": MLPRegressor(max_iter=500)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "MSE": mse,
        "R2 Score": r2
    })

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
print(results_df)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x="R2 Score", y="Model", data=results_df, palette="viridis")
plt.title("Model Comparison by R2 Score")
plt.xlabel("R2 Score")
plt.ylabel("Model")
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_model_r2 = results_df.iloc[0]['R2 Score']
print(f"The best performing model is {best_model_name} with an R2 Score of {best_model_r2:.4f}")